# RT-DETR on SH17 (PPE): prepare → train → evaluate → package

**Before running, set in the right-hand panel (Settings):**
1. **Accelerator → GPU T4 x2** (this notebook uses one T4; don't use P100, recent PyTorch builds may not support it)
2. **Internet → On** (needs a phone-verified Kaggle account)
3. **Input:** the *SH17 Dataset for PPE Detection* must be attached (it is if you clicked *Create Notebook* on the dataset page)

**Two runs:**
- **Run 1, `SMOKE_TEST = True`**: interactive (*Run All*), ~20–30 min. Checks everything works and estimates the time per epoch.
- **Run 2, `SMOKE_TEST = False`**: *Save Version → Save & Run All (Commit)*. Runs in the background, so you can close the browser. Download `artifacts.zip` from the **Output** tab when it's done.

In [ ]:
# ---------------- settings ----------------
REPO_URL   = "https://github.com/Chan-dev12/rtdetr-vision-api.git"
SMOKE_TEST = True     # True: quick end-to-end check.  False: the real training run (use Save & Run All)
TIME_HOURS = 9.0      # hard training budget; Kaggle kills sessions at 12 h, this leaves time for evaluation
EPOCHS     = 60       # upper bound; with TIME_HOURS set, Ultralytics fits the schedule to the time budget
BATCH      = 8        # T4 16 GB at 640 px. Use 4 if you hit CUDA out-of-memory
DEVICE     = "0"

import os
KAGGLE_INPUT = os.environ.get("KAGGLE_INPUT", "/kaggle/input")
WORK         = os.environ.get("WORK", "/kaggle/working")
REPO_DIR     = os.path.join(WORK, "rtdetr-vision-api")

In [ ]:
import glob, json, re, shutil, subprocess, sys, time
from pathlib import Path

def sh(cmd, cwd=None):
    """Run a shell command and stream its output into the notebook."""
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"command failed: {cmd}")

try:
    sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv")
except RuntimeError:
    print("WARNING: no GPU visible. Set Accelerator = GPU T4 x2 in Settings.")

## 1. Get the code

In [ ]:
if Path(REPO_DIR).exists():
    shutil.rmtree(REPO_DIR)
local = [p for p in glob.glob(f"{KAGGLE_INPUT}/**/scripts/prepare_data.py", recursive=True)]
if REPO_URL:
    sh(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
elif local:                      # fallback: repo uploaded as a Kaggle dataset
    shutil.copytree(Path(local[0]).parents[1], REPO_DIR)
else:
    raise SystemExit("Set REPO_URL, or attach the repo as a Kaggle dataset.")
os.chdir(REPO_DIR)
sh("git log -1 --oneline || true")

# Kaggle already has a CUDA build of torch: install only what's missing. Don't reinstall torch.
pip = f"{sys.executable} -m pip install -q ultralytics==8.4.146 imagehash==4.3.2"
try:
    sh(pip)
except RuntimeError:             # some environments need this flag; Kaggle normally doesn't
    sh(pip + " --break-system-packages")
sh(f"{sys.executable} -c \"import torch, ultralytics; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| ultralytics', ultralytics.__version__)\"")

## 2. Find SH17 and check the label format

In [ ]:
hits = [p for p in glob.glob(f"{KAGGLE_INPUT}/**/val_files.txt", recursive=True)
        if (Path(p).parent / "images").is_dir() and (Path(p).parent / "labels").is_dir()]
assert hits, f"SH17 not found under {KAGGLE_INPUT}. Attach the dataset via '+ Add Input'."
SH17 = Path(hits[0]).parent
print("SH17 at:", SH17)
print("images:", len(os.listdir(SH17 / "images")), "| labels:", len(os.listdir(SH17 / "labels")),
      "| official test list:", sum(1 for l in open(SH17 / "val_files.txt") if l.strip()))

# Every label line must be: class_id cx cy w h, with ids 0..16 (sh17.yaml order)
from collections import Counter
ids, bad = Counter(), 0
for f in sorted(glob.glob(str(SH17 / "labels" / "*.txt")))[:500]:
    for line in open(f):
        v = line.split()
        if not v: continue                      # blank line
        if len(v) != 5: bad += 1; continue
        ids[int(float(v[0]))] += 1
print("class ids in first 500 label files:", dict(sorted(ids.items())), "| malformed lines:", bad)
assert max(ids) <= 16, "class ids above 16: label files don't match sh17.yaml. Stop and investigate."

## 3. Build the dataset (downscale to 1280 px, official test split, leak check)

In [ ]:
import yaml
cfg = yaml.safe_load(open("configs/dataset.yaml"))
cfg["sources"][0]["path"] = str(SH17)
cfg["fixed_test"]["list"] = str(SH17 / "val_files.txt")
yaml.safe_dump(cfg, open("configs/dataset_kaggle.yaml", "w"), sort_keys=False)

t = time.time()
sh("python scripts/prepare_data.py --config configs/dataset_kaggle.yaml")
print(f"prepare took {(time.time() - t) / 60:.1f} min")
print(json.dumps(json.load(open("data/processed/prepare_report.json"))["fixed_test"], indent=2))

## 4. Audit + **look at the label previews** (class ids right? is a head under a helmet also labelled `head`?)

In [ ]:
sh("python scripts/audit_data.py --data data/processed/data.yaml --out reports/data_audit.md --preview 12")
from IPython.display import Image as IPImage, display
for f in sorted(glob.glob("reports/label_preview/*.jpg"))[:6]:
    display(IPImage(filename=f, width=900))

## 5. Train

In [ ]:
if SMOKE_TEST:
    overrides = f"epochs=1 fraction=0.05 batch={BATCH} device={DEVICE} name=smoke exist_ok=true"
else:
    overrides = f"epochs={EPOCHS} time={TIME_HOURS} batch={BATCH} device={DEVICE}"
t = time.time()
sh(f"python scripts/train.py --config configs/train.yaml --set {overrides}")
summary = json.load(open("weights/run_summary.json"))
print("training wall-clock:", summary["train_time_human"], "| epochs:", summary["epochs_completed"],
      "| gpu:", summary["environment"]["gpus"])
if SMOKE_TEST:
    per_epoch_min = summary["train_time_seconds"] / 0.05 / 60      # very rough: 5% of the data, 1 epoch
    print(f"ROUGH full-data epoch estimate: {per_epoch_min:.0f} min "
          f"-> about {TIME_HOURS * 60 / per_epoch_min:.0f} epochs fit in TIME_HOURS={TIME_HOURS}")

## 6. Evaluate on the official SH17 test split

In [ ]:
sh(f"python scripts/evaluate.py --weights weights/best.pt --data data/processed/data.yaml --split test "
   f"--conf 0.5 --tag test_at_conf050 --device {DEVICE}")
best_conf = json.load(open("reports/test_at_conf050/metrics.json"))["f1_optimal_conf"]
print("F1-optimal confidence threshold:", best_conf)

# final report at the operating threshold the API will use (per_class.csv feeds the /ask guardrail)
sh(f"python scripts/evaluate.py --weights weights/best.pt --data data/processed/data.yaml --split test "
   f"--conf {best_conf} --tag test --device {DEVICE}")
txt = open("configs/classes.yaml").read()
txt = re.sub(r"operating_conf: [0-9.]+", f"operating_conf: {best_conf}", txt, count=1)
if best_conf <= 0.3:   # keep the "uncertain" band below the operating threshold
    txt = re.sub(r"uncertain_conf: [0-9.]+", f"uncertain_conf: {round(best_conf / 2, 2)}", txt, count=1)
open("configs/classes.yaml", "w").write(txt)
print("configs/classes.yaml ->", re.findall(r"(operating_conf|uncertain_conf): ([0-9.]+)", txt))

In [ ]:
import pandas as pd
m = json.load(open("reports/test/metrics.json"))
print({k: m[k] for k in ("coco_map50", "coco_map50_95", "operating_conf", "operating_precision", "operating_recall")})
display(pd.read_csv("reports/test/per_class.csv"))
display(pd.read_csv("reports/test/size_breakdown.csv"))
display(pd.read_csv("reports/test/confusion_matrix.csv"))
print(m["error_type_counts"])
for f in sorted(glob.glob("reports/test/failures/*.jpg"))[:4]:
    display(IPImage(filename=f, width=900))

## 7. Package everything for download (Output tab → `artifacts.zip`)

In [ ]:
run_dir = Path(summary["run_dir"])
pkg = Path(WORK) / "artifacts"
if pkg.exists():
    shutil.rmtree(pkg)
for src, dst in [("weights", "weights"), ("reports", "reports"), ("configs", "configs")]:
    shutil.copytree(src, pkg / dst)
(pkg / "data").mkdir(parents=True)
for f in ("manifest.csv", "prepare_report.json", "data.yaml"):
    shutil.copy(f"data/processed/{f}", pkg / "data" / f)
(pkg / "run").mkdir()
for f in list(run_dir.glob("*.csv")) + list(run_dir.glob("*.png")) + list(run_dir.glob("*.jpg")) + list(run_dir.glob("*.yaml")):
    shutil.copy(f, pkg / "run" / f.name)
shutil.make_archive(str(Path(WORK) / "artifacts"), "zip", pkg)
shutil.rmtree(pkg)

# keep the saved output small: the processed images can be rebuilt from the manifest
shutil.rmtree("data/processed", ignore_errors=True)
shutil.rmtree(run_dir / "weights", ignore_errors=True)
print("artifacts.zip:", round(os.path.getsize(Path(WORK) / "artifacts.zip") / 1e6, 1), "MB")
print("weights sha256:", summary["weights_sha256"])